In [3]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    confusion_matrix, RocCurveDisplay, roc_auc_score
)


# ----------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------

df = pd.read_csv('Loan_default.csv')


print("Data loaded. Shape:", df.shape)

# ----------------------------------------------------
# 2. TARGET + DROP USELESS COLUMNS
# ----------------------------------------------------
target = "Default"
df = df.drop(columns=["LoanID"])     # ID column not useful

# Drop rows with missing target
df = df.dropna(subset=[target])

# Check for missing values
print("\nMissing Values Count:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)
print("\nDescriptive stats:")
print(df.describe())

Data loaded. Shape: (255347, 18)

Missing Values Count:
Age               0
Income            0
LoanAmount        0
CreditScore       0
MonthsEmployed    0
NumCreditLines    0
InterestRate      0
LoanTerm          0
DTIRatio          0
Education         0
EmploymentType    0
MaritalStatus     0
HasMortgage       0
HasDependents     0
LoanPurpose       0
HasCoSigner       0
Default           0
dtype: int64

Data Types:
Age                 int64
Income              int64
LoanAmount          int64
CreditScore         int64
MonthsEmployed      int64
NumCreditLines      int64
InterestRate      float64
LoanTerm            int64
DTIRatio          float64
Education          object
EmploymentType     object
MaritalStatus      object
HasMortgage        object
HasDependents      object
LoanPurpose        object
HasCoSigner        object
Default             int64
dtype: object

Descriptive stats:
                 Age         Income     LoanAmount    CreditScore  \
count  255347.000000  255347.0000

In [4]:
binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']

binary_map = {"yes": 1, "no": 0, "y": 1, "n": 0, "1": 1, "0": 0}

for col in binary_cols:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.lower().map(binary_map)


X = df.drop(columns=[target])
y = df[target]
print(df.head())
# ----------------------------------------------------
# 3. DEFINE COLUMN TYPES
# ----------------------------------------------------
categorical_cols =  ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']

numerical_cols= ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate',
                 'LoanTerm', 'DTIRatio', 'HasDependents', 'HasCoSigner', 'HasMortgage']



print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)


   Age  Income  LoanAmount  CreditScore  MonthsEmployed  NumCreditLines  \
0   56   85994       50587          520              80               4   
1   69   50432      124440          458              15               1   
2   46   84208      129188          451              26               3   
3   32   31713       44799          743               0               3   
4   60   20437        9139          633               8               4   

   InterestRate  LoanTerm  DTIRatio    Education EmploymentType MaritalStatus  \
0         15.23        36      0.44   Bachelor's      Full-time      Divorced   
1          4.81        60      0.68     Master's      Full-time       Married   
2         21.17        24      0.31     Master's     Unemployed      Divorced   
3          7.07        24      0.23  High School      Full-time       Married   
4          6.51        48      0.73   Bachelor's     Unemployed      Divorced   

   HasMortgage  HasDependents LoanPurpose  HasCoSigner  Defaul

In [5]:

# ----------------------------------------------------
# 4. PREPROCESSING PIPELINE
# ----------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numerical_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ]
)

# ----------------------------------------------------
# 5. TRAIN / TEST SPLIT
# ----------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ----------------------------------------------------
# 6. BASELINE MODELS
# ----------------------------------------------------
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1)
}

results = {}
roc_curves = {}


In [6]:

# ----------------------------------------------------
# 7. TRAINING + METRICS
# ----------------------------------------------------
for name, model in models.items():

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred)

    results[name] = (acc, f1, rec, auc, prec)
    roc_curves[name] = (y_test, y_proba)

    # ---------------- Confusion Matrix ----------------
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots()
    ax.imshow(cm, cmap="Blues")
    ax.set_title(f"Confusion Matrix – {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha='center', va='center')

    os.makedirs("outputs", exist_ok=True)
    plt.savefig(f"outputs/confusion_matrix_{name}.png", dpi=150)
    plt.close()


In [7]:

# ----------------------------------------------------
# 8. ROC CURVE
# ----------------------------------------------------
fig, ax = plt.subplots()

for name, (yt, yp) in roc_curves.items():
    RocCurveDisplay.from_predictions(yt, yp, name=name, ax=ax)

ax.set_title("ROC Curve - Logistic Regression vs Random Forest")

plt.savefig("outputs/roc_curve.png", dpi=150)
plt.close()

# ----------------------------------------------------
# 9. PRINT SUMMARY
# ----------------------------------------------------
print("\n===== BASELINE MODEL RESULTS =====\n")
for name, (acc, f1, rec, auc,prec) in results.items():
    print(f"{name}:")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1 Score : {f1:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  ROC-AUC  : {auc:.4f}\n")
    print(f"  Precision  : {prec:.4f}\n")

print("➡️ Confusion matrices and ROC curve saved in ./outputs/")



===== BASELINE MODEL RESULTS =====

LogisticRegression:
  Accuracy : 0.6775
  F1 Score : 0.3356
  Recall   : 0.7015
  ROC-AUC  : 0.7543

  Precision  : 0.2206

Decision Tree:
  Accuracy : 0.8039
  F1 Score : 0.2139
  Recall   : 0.2297
  ROC-AUC  : 0.5545

  Precision  : 0.2001

Random Forest:
  Accuracy : 0.8853
  F1 Score : 0.0565
  Recall   : 0.0295
  ROC-AUC  : 0.7348

  Precision  : 0.6366

➡️ Confusion matrices and ROC curve saved in ./outputs/
